In [ ]:
!pip install -q pymupdf sentence-transformers transformers torch nltk pandas

In [ ]:
import fitz
import os
import nltk
import pandas as pd

from google.colab import files

nltk.download('punkt')
nltk.download('punkt_tab')

# Upload PDF
uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]

print("Uploaded PDF:", pdf_path)

# Extract text page by page
doc = fitz.open(pdf_path)

pages = []

for page_num, page in enumerate(doc):
    text = page.get_text("text")

    if text.strip():
        pages.append({
            "page": page_num + 1,
            "text": text
        })

print("Total PDF Pages:", len(doc))
print("Pages with extracted text:", len(pages))

print("\nSample extracted text:")
print(pages[0]["text"][:1000])

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Saving IMP.pdf to IMP (1).pdf
Uploaded PDF: IMP (1).pdf
Total PDF Pages: 27
Pages with extracted text: 27

Sample extracted text:
1. Life Cycle of Generative AI 
Generative AI is a type of artificial intelligence that learns patterns from existing data and generates 
new content such as text, images, audio, video or code. 
Life Cycle of Generative AI 
Data Collection 
      ↓ 
Data Preparation 
      ↓ 
Model Selection 
      ↓ 
Pre-training 
      ↓ 
Fine-tuning 
      ↓ 
Prompt Engineering 
      ↓ 
Generation / Inference 
      ↓ 
Evaluation 
      ↓ 
Deployment & Monitoring 
1. Data Collection 
Large amounts of data such as text, images, audio or code are collected from suitable sources. 
2. Data Preparation 
The collected data is cleaned, normalized and converted into a suitable format. For text, tokenization is 
performed. 
3. Model Selection 
An appropriate generative model is selected, such as: 
• 
Transformer 
• 
GAN 
• 
VAE 
• 
Diffusion model 
4. Pre-training 



In [ ]:
def create_chunks(pages, chunk_size=1200):

    chunks = []

    for page in pages:

        paragraphs = page["text"].split("\n")

        current_chunk = ""

        for paragraph in paragraphs:

            paragraph = paragraph.strip()

            if not paragraph:
                continue

            if len(current_chunk) + len(paragraph) < chunk_size:

                current_chunk += paragraph + " "

            else:

                if current_chunk.strip():

                    chunks.append({
                        "text": current_chunk.strip(),
                        "page": page["page"]
                    })

                current_chunk = paragraph + " "

        if current_chunk.strip():

            chunks.append({
                "text": current_chunk.strip(),
                "page": page["page"]
            })

    return chunks


chunks = create_chunks(pages)

print("Total chunks created:", len(chunks))

for i, chunk in enumerate(chunks[:3]):

    print(f"\nChunk {i+1}")
    print("Page:", chunk["page"])
    print(chunk["text"][:300])

Total chunks created: 35

Chunk 1
Page: 1
1. Life Cycle of Generative AI Generative AI is a type of artificial intelligence that learns patterns from existing data and generates new content such as text, images, audio, video or code. Life Cycle of Generative AI Data Collection ↓ Data Preparation ↓ Model Selection ↓ Pre-training ↓ Fine-tunin

Chunk 2
Page: 2
The model learns general patterns and relationships from a large dataset. 5. Fine-tuning The pre-trained model is trained further on a specific dataset to improve its performance for a particular task. 6. Prompt Engineering Suitable instructions or prompts are designed to obtain the required output 

Chunk 3
Page: 2
↓ Numerical Vectors


In [ ]:
def create_chunks(pages, chunk_size=1200):

    chunks = []

    for page in pages:

        paragraphs = page["text"].split("\n")

        current_chunk = ""

        for paragraph in paragraphs:

            paragraph = paragraph.strip()

            if not paragraph:
                continue

            if len(current_chunk) + len(paragraph) < chunk_size:

                current_chunk += paragraph + " "

            else:

                if current_chunk.strip():

                    chunks.append({
                        "text": current_chunk.strip(),
                        "page": page["page"]
                    })

                current_chunk = paragraph + " "

        if current_chunk.strip():

            chunks.append({
                "text": current_chunk.strip(),
                "page": page["page"]
            })

    return chunks


chunks = create_chunks(pages)

print("Total chunks created:", len(chunks))

for i, chunk in enumerate(chunks[:3]):

    print(f"\nChunk {i+1}")
    print("Page:", chunk["page"])
    print(chunk["text"][:300])

Total chunks created: 35

Chunk 1
Page: 1
1. Life Cycle of Generative AI Generative AI is a type of artificial intelligence that learns patterns from existing data and generates new content such as text, images, audio, video or code. Life Cycle of Generative AI Data Collection ↓ Data Preparation ↓ Model Selection ↓ Pre-training ↓ Fine-tunin

Chunk 2
Page: 2
The model learns general patterns and relationships from a large dataset. 5. Fine-tuning The pre-trained model is trained further on a specific dataset to improve its performance for a particular task. 6. Prompt Engineering Suitable instructions or prompts are designed to obtain the required output 

Chunk 3
Page: 2
↓ Numerical Vectors


In [ ]:
import torch

from sentence_transformers import SentenceTransformer
from sentence_transformers import util

# Load embedding model
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# Extract chunk text
chunk_texts = [chunk["text"] for chunk in chunks]

# Generate PDF embeddings
document_embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_tensor=True,
    show_progress_bar=True
)

def retrieve_evidence(claim, top_k=4):

    claim_embedding = embedding_model.encode(
        claim,
        convert_to_tensor=True
    )

    similarities = util.cos_sim(
        claim_embedding,
        document_embeddings
    )[0]

    top_results = torch.topk(
        similarities,
        k=min(top_k, len(chunks))
    )

    evidence = []

    for score, idx in zip(
        top_results.values,
        top_results.indices
    ):

        idx = idx.item()

        evidence.append({
            "text": chunks[idx]["text"],
            "page": chunks[idx]["page"],
            "similarity": float(score)
        })

    return evidence


# Test retrieval
test_claim = "Hallucination occurs when an AI model generates incorrect information."

results = retrieve_evidence(test_claim)

for result in results:

    print("\nPage:", result["page"])
    print("Similarity:", round(result["similarity"], 3))
    print(result["text"][:400])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]


Page: 22
Similarity: 0.688
Methods such as: • Mixture of Experts (MoE) • Quantization • Knowledge distillation • Efficient attention can improve efficiency. Scaling relationship A simplified view is: 𝑃𝑒𝑟𝑓𝑜𝑟𝑚𝑎𝑛𝑐𝑒↑ as 𝑀𝑜𝑑𝑒𝑙 𝑆𝑖𝑧𝑒, 𝐷𝑎𝑡𝑎, 𝐶𝑜𝑚𝑝𝑢𝑡𝑒↑ However, simply making a model bigger does not guarantee better results; data quality, training strategy and architecture also matter. 19. How Does Hallucination Happen in Generative A

Page: 23
Similarity: 0.585
Possible False Information The model primarily learns to predict likely sequences of tokens. It does not automatically verify every generated statement against reality. Major causes 1. Incomplete or noisy training data 2. Ambiguous prompts 3. Lack of reliable external knowledge 4. Outdated information 5. Incorrect information in training data 6. Long or complex reasoning chains 7. Overconfident ge

Page: 1
Similarity: 0.402
1. Life Cycle of Generative AI Generative AI is a type of artificial intelligence that learns patterns from existin

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

import torch.nn.functional as F

MODEL_NAME = "cross-encoder/nli-deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

nli_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

nli_model.to(device)
nli_model.eval()

# Read model labels
id2label = nli_model.config.id2label

print("Device:", device)
print("Model labels:", id2label)


# Map model labels to NLI categories
label_map = {}

for idx, label in id2label.items():

    label = label.lower()

    if "entail" in label:
        label_map["entailment"] = int(idx)

    elif "contrad" in label:
        label_map["contradiction"] = int(idx)

    elif "neutral" in label:
        label_map["neutral"] = int(idx)


# Fallback for the standard DeBERTa NLI label ordering
if len(label_map) != 3:

    label_map = {
        "contradiction": 0,
        "entailment": 1,
        "neutral": 2
    }

print("Final label mapping:", label_map)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Device: cpu
Model labels: {0: 'contradiction', 1: 'entailment', 2: 'neutral'}
Final label mapping: {'contradiction': 0, 'entailment': 1, 'neutral': 2}


In [ ]:
def verify_claim(claim, evidence_list):

    pairs = []

    for evidence in evidence_list:

        premise = evidence["text"]

        hypothesis = claim

        pairs.append((premise, hypothesis))

    encoded = tokenizer(
        [p[0] for p in pairs],
        [p[1] for p in pairs],
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )

    encoded = {
        key: value.to(device)
        for key, value in encoded.items()
    }

    with torch.no_grad():

        outputs = nli_model(**encoded)

        probabilities = F.softmax(
            outputs.logits,
            dim=-1
        )

    # Get the best evidence for each class
    entailment_scores = probabilities[
        :, label_map["entailment"]
    ]

    contradiction_scores = probabilities[
        :, label_map["contradiction"]
    ]

    neutral_scores = probabilities[
        :, label_map["neutral"]
    ]

    best_entailment_idx = torch.argmax(
        entailment_scores
    ).item()

    best_contradiction_idx = torch.argmax(
        contradiction_scores
    ).item()

    max_entailment = float(
        entailment_scores.max()
    )

    max_contradiction = float(
        contradiction_scores.max()
    )

    max_neutral = float(
        neutral_scores.max()
    )

    # Classification thresholds
    if (
        max_entailment >= 0.65
        and max_entailment - max_contradiction >= 0.10
    ):

        status = "SUPPORTED"

        best_idx = best_entailment_idx

    elif (
        max_contradiction >= 0.65
        and max_contradiction - max_entailment >= 0.10
    ):

        status = "CONTRADICTED"

        best_idx = best_contradiction_idx

    else:

        status = "UNVERIFIED"

        best_idx = torch.argmax(
            neutral_scores
        ).item()

    return {

        "claim": claim,

        "status": status,

        "entailment": round(max_entailment, 4),

        "contradiction": round(max_contradiction, 4),

        "neutral": round(max_neutral, 4),

        "evidence": evidence_list[best_idx]["text"],

        "page": evidence_list[best_idx]["page"]

    }

In [ ]:
from nltk.tokenize import sent_tokenize

def extract_claims(answer):

    sentences = sent_tokenize(answer)

    claims = []

    for sentence in sentences:

        sentence = sentence.strip()

        if len(sentence.split()) >= 4:

            claims.append(sentence)

    return claims


# Example
sample_answer = """
Hallucination occurs when an AI model generates
incorrect or unsupported information.

Transformers use self-attention to process
relationships between tokens.

The first Transformer was invented in 1995
by John Smith.
"""

claims = extract_claims(sample_answer)

print("Extracted claims:")

for i, claim in enumerate(claims):

    print(f"{i+1}. {claim}")

Extracted claims:
1. Hallucination occurs when an AI model generates
incorrect or unsupported information.
2. Transformers use self-attention to process
relationships between tokens.
3. The first Transformer was invented in 1995
by John Smith.


In [ ]:
def detect_hallucinations(
    generated_answer,
    top_k=4
):

    claims = extract_claims(generated_answer)

    results = []

    print("\nStarting Hallucination Detection...\n")

    for i, claim in enumerate(claims):

        print(f"Checking Claim {i+1}...")

        # Step 1: Retrieve relevant PDF passages
        evidence = retrieve_evidence(
            claim,
            top_k=top_k
        )

        # Step 2: Verify using NLI
        verification = verify_claim(
            claim,
            evidence
        )

        results.append(verification)

    return results


def display_results(results):

    df = pd.DataFrame(results)

    total = len(df)

    supported = sum(
        df["status"] == "SUPPORTED"
    )

    contradicted = sum(
        df["status"] == "CONTRADICTED"
    )

    unverified = sum(
        df["status"] == "UNVERIFIED"
    )

    support_rate = (
        supported / total * 100
        if total else 0
    )

    unsupported_rate = (
        (contradicted + unverified) / total * 100
        if total else 0
    )

    print("\n" + "=" * 55)
    print("       HALLUCINATION DETECTION REPORT")
    print("=" * 55)

    print("Total Claims:", total)

    print("Supported Claims:", supported)

    print("Contradicted Claims:", contradicted)

    print("Unverified Claims:", unverified)

    print(
        f"Evidence Support Rate: {support_rate:.2f}%"
    )

    print(
        f"Unsupported Claim Rate: {unsupported_rate:.2f}%"
    )

    print("=" * 55)

    display(
        df[
            [
                "claim",
                "status",
                "entailment",
                "contradiction",
                "neutral",
                "page"
            ]
        ]
    )

    print("\nDETAILED EVIDENCE")

    for i, row in df.iterrows():

        print("\n" + "-" * 55)

        print("Claim:", row["claim"])

        print("Status:", row["status"])

        print("Reference Page:", row["page"])

        print("Evidence:", row["evidence"])

    return df

In [ ]:
# Enter user question
user_question = input("Enter your question: ")

# Enter LLM-generated answer
generated_answer = input(
    "\nPaste the LLM-generated answer: "
)

print("\nUSER QUESTION:")
print(user_question)

print("\nGENERATED ANSWER:")
print(generated_answer)

# Run hallucination detection
results = detect_hallucinations(
    generated_answer,
    top_k=4
)

# Display final report
final_report = display_results(results)

Enter your question: What is hallucination in Generative AI?

Paste the LLM-generated answer: Hallucination occurs when a generative AI model produces information that is incorrect, unsupported or fabricated but appears convincing.

USER QUESTION:
What is hallucination in Generative AI?

GENERATED ANSWER:
Hallucination occurs when a generative AI model produces information that is incorrect, unsupported or fabricated but appears convincing.

Starting Hallucination Detection...

Checking Claim 1...

       HALLUCINATION DETECTION REPORT
Total Claims: 1
Supported Claims: 0
Contradicted Claims: 0
Unverified Claims: 1
Evidence Support Rate: 0.00%
Unsupported Claim Rate: 100.00%


,claim,status,entailment,contradiction,neutral,page
0,Hallucination occurs when a generative AI mode...,UNVERIFIED,0.1802,0.0511,0.9994,1



DETAILED EVIDENCE

-------------------------------------------------------
Claim: Hallucination occurs when a generative AI model produces information that is incorrect, unsupported or fabricated but appears convincing.
Status: UNVERIFIED
Reference Page: 1
Evidence: 1. Life Cycle of Generative AI Generative AI is a type of artificial intelligence that learns patterns from existing data and generates new content such as text, images, audio, video or code. Life Cycle of Generative AI Data Collection ↓ Data Preparation ↓ Model Selection ↓ Pre-training ↓ Fine-tuning ↓ Prompt Engineering ↓ Generation / Inference ↓ Evaluation ↓ Deployment & Monitoring 1. Data Collection Large amounts of data such as text, images, audio or code are collected from suitable sources. 2. Data Preparation The collected data is cleaned, normalized and converted into a suitable format. For text, tokenization is performed. 3. Model Selection An appropriate generative model is selected, such as: • Transformer • GAN •

In [ ]:
# Enter user question
user_question = input("Enter your question: ")

# Enter LLM-generated answer
generated_answer = input(
    "\nPaste the LLM-generated answer: "
)

print("\nUSER QUESTION:")
print(user_question)

print("\nGENERATED ANSWER:")
print(generated_answer)

# Run hallucination detection
results = detect_hallucinations(
    generated_answer,
    top_k=4
)

# Display final report
final_report = display_results(results)

Enter your question: Is GPT an encoder-only Transformer?

Paste the LLM-generated answer: Yes, GPT is an encoder-only Transformer architecture.

USER QUESTION:
Is GPT an encoder-only Transformer?

GENERATED ANSWER:
Yes, GPT is an encoder-only Transformer architecture.

Starting Hallucination Detection...

Checking Claim 1...

       HALLUCINATION DETECTION REPORT
Total Claims: 1
Supported Claims: 0
Contradicted Claims: 1
Unverified Claims: 0
Evidence Support Rate: 0.00%
Unsupported Claim Rate: 100.00%


,claim,status,entailment,contradiction,neutral,page
0,"Yes, GPT is an encoder-only Transformer archit...",CONTRADICTED,0.3347,0.9997,0.9842,21



DETAILED EVIDENCE

-------------------------------------------------------
Claim: Yes, GPT is an encoder-only Transformer architecture.
Status: CONTRADICTED
Reference Page: 21
Evidence: Transformers provide excellent performance but their computational cost, memory requirements, hallucinations and training requirements are important limitations. 18. How Can a Transformer Model Be Scaled? Scaling a Transformer means increasing its ability and capacity by increasing model size, training data, computation, or context length. Main scaling factors Transformer Scaling ↓ ┌───────────┼───────────┐ ↓           ↓           ↓ Parameters     Data      Computation ↓           ↓           ↓ More layers   More data   More training More neurons              FLOPs 1. Increase Model Parameters The number of: • layers • hidden dimensions • attention heads can be increased. 2. Increase Training Data Training on more high-quality and diverse data can improve the model's generalization. 3. Increase Computa

In [ ]:
# Enter user question
user_question = input("Enter your question: ")

# Enter LLM-generated answer
generated_answer = input(
    "\nPaste the LLM-generated answer: "
)

print("\nUSER QUESTION:")
print(user_question)

print("\nGENERATED ANSWER:")
print(generated_answer)

# Run hallucination detection
results = detect_hallucinations(
    generated_answer,
    top_k=4
)

# Display final report
final_report = display_results(results)

Enter your question: What is the exact accuracy of BERT according to the reference PDF?

Paste the LLM-generated answer: BERT achieves 98.7% accuracy according to the reference document.

USER QUESTION:
What is the exact accuracy of BERT according to the reference PDF?

GENERATED ANSWER:
BERT achieves 98.7% accuracy according to the reference document.

Starting Hallucination Detection...

Checking Claim 1...

       HALLUCINATION DETECTION REPORT
Total Claims: 1
Supported Claims: 0
Contradicted Claims: 0
Unverified Claims: 1
Evidence Support Rate: 0.00%
Unsupported Claim Rate: 100.00%


,claim,status,entailment,contradiction,neutral,page
0,BERT achieves 98.7% accuracy according to the ...,UNVERIFIED,0.0006,0.009,0.9996,8



DETAILED EVIDENCE

-------------------------------------------------------
Claim: BERT achieves 98.7% accuracy according to the reference document.
Status: UNVERIFIED
Reference Page: 8
Evidence: • Named Entity Recognition • Sentence similarity Example: BERT → Encoder-only GPT  → Decoder-only T5   → Encoder-decoder 7. Encoder-Decoder Transformer An encoder-decoder Transformer contains two major components: 1. Encoder 2. Decoder It is commonly used for sequence-to-sequence tasks such as machine translation. Architecture INPUT ↓ ┌─────────────┐ │   ENCODER   │ │ Self-Attn.  │ │ Feed Forward│ └──────┬──────┘ │ ↓ Encoded Context │ ↓ ┌─────────────┐ │   DECODER   │ │ Masked Attn.│ │ Cross Attn. │ │ Feed Forward│ └──────┬──────┘ ↓


In [ ]:
output_file = "hallucination_detection_report.xlsx"

final_report.to_excel(
    output_file,
    index=False
)

files.download(output_file)

print("Report downloaded successfully!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Report downloaded successfully!
